<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/LightBGM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#均線糾結並升級為Lightbgm優化模式
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【新增：指定光學鏡頭、先進設備與集團概念股】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【新增：工業電腦 (IPC) 相關概念股】 ---
    "2395.TW": "研華",
    "8050.TWO": "廣積",
    "6525.TW": "捷迅",  # 若需純IPC可換為融程電等，此處列主流代表
    "3522.TWO": "御寶/遠端等(暫略)",  # 換成更標準的如：
    "2393.TW": "億光",
    "6166.TW": "凌華",
    "2465.TW": "麗臺",
    "5536.TWO": "聖暉*",
    # (註：以下精選主流工業電腦代表)
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "5251.TWO": "龍燈-KY",  # 換即時IPC
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    # --- 【新增：電動車 (EV) 與車用電子相關概念股】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "3665.TW": "貿聯-KY",  # 重複確認
    "2308.TW": "台達電",  # 重複確認
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    "2421.TW": "建準",
    # --- 【21-33. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    # --- 【34. 銅箔與PCB上游材料】 ---
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件..."
)
predictions = []

all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 計算篩選條件所需的指標（直接使用當天實際數據，不作 shift）
    vol_mean_5 = df["Volume"].rolling(5).mean()
    ma5 = df["Close"].rolling(5).mean()
    ma10 = df["Close"].rolling(10).mean()
    ma20 = df["Close"].rolling(20).mean()
    ma_max = pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
    ma_min = pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)

    # 條件 1：5日均量 >= 500 張 (500,000 股)
    liquidity_ok = vol_mean_5 >= 500000
    # 條件 2：5日線、10日線、20日線差距在 3% 以內
    ma_tangle = (ma_max - ma_min) / (ma_min + 1e-6) <= 0.03

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      # 取得當前預測日期的 Index 標籤
      target_idx = df_clean.index[i]

      # 套用篩選條件：若當天不符合流動性或均線糾結，則略過
      if not liquidity_ok.loc[target_idx] or not ma_tangle.loc[target_idx]:
        continue

      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = target_idx.strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(
      " 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)"
  )
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前最近 5 個交易日內，沒有符合「5日均量 >= 500張 且 均線糾結 3%」的股票。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件...

 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)
| 預測日期   | 股票名稱   |   股票代號 | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:-----------|-----------:|:---------|:--------------|:--------------|
| 2026-08-14 | 矽格       |       6257 | 25.65%   | 46.29%        | 1.8x          |
| 2026-08-14 | 臻鼎-KY    |       4958 | 31.83%   | 44.05%        | 1.38x         |
| 2026-08-14 | 辛耘       |       3583 | 25.42%   | 34.96%        | 1.38x         |
| 2026-08-14 | 華泰       |       2329 | 28.98%   | 32.73%        | 1.13x         |
| 2026-08-14 | 鈦昇       |       8027 | 29.45%   | 31.65%        | 1.07x         |
| 2026-08-14 | 京元電子   |       2449 | 32.07%   | 31.31%        | 0.98x         |
| 2026-08-14 | 佳必琪     |       6197 | 23.75%   | 30.01%        | 1.26x         |
| 2026-08-14 | 立敦       |       6175 | 30.64%   | 28.18%        | 0.92x         |
| 2026-08-14 | 新應材     |       4749 | 27.79%   | 27.82%        | 1.0x          |
| 2026-0

In [ ]:
#無均線糾結升級為lightgbm優化模式
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【新增：指定光學鏡頭、先進設備與集團概念股】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【新增：工業電腦 (IPC) 相關概念股】 ---
    "2395.TW": "研華",
    "8050.TWO": "廣積",
    "6525.TW": "捷迅",  # 若需純IPC可換為融程電等，此處列主流代表
    "3522.TWO": "御寶/遠端等(暫略)",  # 換成更標準的如：
    "2393.TW": "億光",
    "6166.TW": "凌華",
    "2465.TW": "麗臺",
    "5536.TWO": "聖暉*",
    # (註：以下精選主流工業電腦代表)
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "5251.TWO": "龍燈-KY",  # 換即時IPC
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    # --- 【新增：電動車 (EV) 與車用電子相關概念股】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "3665.TW": "貿聯-KY",  # 重複確認
    "2308.TW": "台達電",  # 重複確認
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    "2421.TW": "建準",
    # --- 【21-33. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    # --- 【34. 銅箔與PCB上游材料】 ---
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測..."
)
predictions = []

# 批次下載所有股票資料以大幅提升速度
all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = df_clean.index[i].strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(" 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) ")
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前無法產生預測結果。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測...

 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) 
| 預測日期   | 股票名稱          |   股票代號 | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:------------------|-----------:|:---------|:--------------|:--------------|
| 2026-08-14 | 頎邦              |       6147 | 18.53%   | 71.58%        | 3.86x         |
| 2026-08-14 | 聯亞              |       3081 | 50.83%   | 66.35%        | 1.31x         |
| 2026-08-14 | 南電              |       8046 | 44.66%   | 62.46%        | 1.4x          |
| 2026-08-14 | 台光電            |       2383 | 48.69%   | 57.62%        | 1.18x         |
| 2026-08-14 | 大銀微系統        |       4576 | 31.12%   | 56.99%        | 1.83x         |
| 2026-08-14 | 萬潤              |       6187 | 34.2%    | 56.64%        | 1.66x         |
| 2026-08-14 | 鈺創              |       5351 | 36.34%   | 56.59%        | 1.56x         |
| 2026-08-14 | AES-KY            |       6781 | 33.73%   | 56.19%        | 1.67x         |
| 2026-08-14 

In [1]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典 (包含您指定的所有完整擴充股票池)
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 磁性元件 / 網通高速連接器】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. ASIC / 晶片設計】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. 電源供應器 / 伺服器電源】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力與重電儲能】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人概念】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 指定光學鏡頭與高階設備】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【22. 工業電腦 (IPC)】 ---
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    # --- 【23. 電動車 (EV) 與車用電子】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    # --- 【24-34. 其他族群與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "6770.TW": "力積電",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "1773.TW": "勝一",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "8358.TWO": "金居",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe
  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  d["NATR"] = tr.rolling(14).mean() / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  d["BB_Bandwidth"] = ((ma20 + 2 * std20) - (ma20 - 2 * std20)) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0
  d["Alpha_5d"] = d["Close"].pct_change(5) - market_df["Close"].pct_change(5)

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，執行 2026/04/01 ~ 2026/06/30"
    " 歷史回測驗證..."
)
backtest_results = []

all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="3y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 計算條件指標
    vol_mean_5 = df["Volume"].rolling(5).mean()
    ma5 = df["Close"].rolling(5).mean()
    ma10 = df["Close"].rolling(10).mean()
    ma20 = df["Close"].rolling(20).mean()
    ma_max = pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
    ma_min = pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)

    liquidity_ok = vol_mean_5 >= 500000
    ma_tangle = (ma_max - ma_min) / (ma_min + 1e-6) <= 0.03

    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    df_feat, feature_cols = compute_features(df, market_df)
    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])

    # 鎖定回測區間：2026年4月1日至2026年6月30日
    mask_date = (df_clean.index >= "2026-04-01") & (
        df_clean.index <= "2026-06-30"
    )
    target_indices = df_clean.index[mask_date]

    for target_idx in target_indices:
      # 安檢哨檢查：流動性與均線糾結
      if not liquidity_ok.loc[target_idx] or not ma_tangle.loc[target_idx]:
        continue

      pos = df_clean.index.get_loc(target_idx)
      if pos < 60:
        continue

      X_train = df_clean[feature_cols].iloc[:pos]
      y_train = df_clean["Target"].iloc[:pos]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )
      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      target_features = df_clean[feature_cols].iloc[[pos]].clip(
          lower_bound, upper_bound, axis=1
      )
      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]

      # 決策規則：預測機率 >= 51% 即觸發訊號 (Y_pred = 1)
      if prob >= 0.51:
        actual_val = df_clean.loc[target_idx, "Target"]
        backtest_results.append({
            "預測日期": target_idx.strftime("%Y-%m-%d"),
            "股票名稱": name,
            "股票代號": ticker.split(".")[0],
            "預測機率": f"{round(float(prob) * 100, 2)}%",
            "raw_prob": float(prob),
            "實際達成(10%漲幅)": int(actual_val),
        })
  except Exception:
    pass

# 成效統計報表
res_df = pd.DataFrame(backtest_results)
if not res_df.empty:
  res_df = res_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  total_signals = len(res_df)
  successful_signals = res_df["實際達成(10%漲幅)"].sum()
  win_rate = (successful_signals / total_signals) * 100

  print("\n" + "=" * 65)
  print(" 📊 2026年 4月 ~ 6月 機器學習選股模型回測成效總結報告")
  print("=" * 65)
  print(f" 總觸發訊號數 (分母): {total_signals} 筆 (機率 >= 51%)")
  print(f" 成功達標數 (分子): {successful_signals} 筆 (未來10日內漲幅 >= 10%)")
  print(f" 實戰預測勝率 (Precision): {round(win_rate, 2)}%")
  print("=" * 65)
  print("\n【詳細觸發訊號清單前 20 筆】")
  print(
      res_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "預測機率",
          "實際達成(10%漲幅)",
      ]]
      .head(20)
      .to_markdown(index=False)
  )
else:
  print("在指定區間內沒有產生符合 51% 門檻的訊號。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，執行 2026/04/01 ~ 2026/06/30 歷史回測驗證...

 📊 2026年 4月 ~ 6月 機器學習選股模型回測成效總結報告
 總觸發訊號數 (分母): 67 筆 (機率 >= 51%)
 成功達標數 (分子): 34 筆 (未來10日內漲幅 >= 10%)
 實戰預測勝率 (Precision): 50.75%

【詳細觸發訊號清單前 20 筆】
| 預測日期   | 股票名稱   |   股票代號 | 預測機率   |   實際達成(10%漲幅) |
|:-----------|:-----------|-----------:|:-----------|--------------------:|
| 2026-06-30 | 智邦       |       2345 | 65.95%     |                   1 |
| 2026-06-29 | 三陽工業   |       2206 | 56.11%     |                   0 |
| 2026-06-26 | 萬潤       |       6187 | 54.09%     |                   0 |
| 2026-06-26 | 光聖       |       6442 | 52.49%     |                   0 |
| 2026-06-25 | 智邦       |       2345 | 68.87%     |                   1 |
| 2026-06-25 | 光聖       |       6442 | 58.44%     |                   0 |
| 2026-06-24 | 智邦       |       2345 | 68.42%     |                   1 |
| 2026-06-24 | 光聖       |       6442 | 55.37%     |                   0 |
| 2026-06-24 | 穩懋       |       3105 | 51.01%     |